# Final Model Comparison — Early Prediction of Prolonged ICU Stay (v1)

**Project:** ICU Patient Care Analysis (MIMIC-IV Clinical Database Demo)  
**Phase:** Assignment 2 — Final Modeling Run  
**Author:** Luis Gonzalez  
**Run location:** `./notebooks`  
**Last updated:** 2026-03-22

## Purpose

This notebook is the **official final modeling run** for Assignment 2. It is meant to be cleaner and more focused than the earlier prototype notebook. The goal is to run the final comparison that matches the locked modeling strategy and to save the artifacts needed for the report and the Show Your Work submission.

The comparison in this notebook is intentionally small:

- majority-class baseline
- logistic regression
- random forest

That is enough for this capstone. The point is not to build the fanciest model possible. The point is to run a fair comparison, keep leakage under control, and generate outputs I can actually explain and defend.

## What this notebook produces

If the notebook runs successfully, it should create artifacts under `../outputs/` including:

### Tables
- `model_v1_split_manifest.csv`
- `model_v1_search_results_logreg.csv`
- `model_v1_search_results_rf.csv`
- `model_v1_cv_metrics_by_fold.csv`
- `model_v1_cv_metrics_summary.csv`
- `model_v1_test_metrics.csv`
- `model_v1_model_comparison.csv`
- `model_v1_test_predictions.csv`
- `model_v1_logreg_coefficients.csv`
- `model_v1_rf_feature_importance.csv`

### Figures
- `model_v1_roc_curve_comparison.png`
- `model_v1_precision_recall_curve_comparison.png`
- `model_v1_confusion_matrix_logreg.png`
- `model_v1_confusion_matrix_rf.png`
- `model_v1_logreg_coefficients.png`
- `model_v1_rf_feature_importance.png`

### Models
- `model_v1_logreg.joblib`
- `model_v1_random_forest.joblib`
- optional preprocessing objects, if saved separately

## Notes

- This notebook assumes it is being run from the `./notebooks` directory.
- It assumes the prepared dataset already exists at `../data/processed/icu_stay_modeling_24h_v1.csv`.
- It keeps related records together at the **subject level** during splitting to reduce leakage across repeated patients.
- Held-out test results are the official results for the final comparison. Cross-validation is used only inside the training portion.


## 1) Setup

In [1]:
from pathlib import Path
import inspect
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    GroupKFold,
    GroupShuffleSplit,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
RANDOM_STATE = 42


## 2) Paths and output locations

In [2]:
DATA_PATH = Path("../data/processed/icu_stay_modeling_24h_v1.csv")

OUTPUTS_DIR = Path("../outputs")
TABLES_DIR = OUTPUTS_DIR / "tables"
FIGURES_DIR = OUTPUTS_DIR / "figures"
MODELS_DIR = OUTPUTS_DIR / "models"

for p in [TABLES_DIR, FIGURES_DIR, MODELS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Data path:", DATA_PATH.resolve())
print("Tables dir:", TABLES_DIR.resolve())
print("Figures dir:", FIGURES_DIR.resolve())
print("Models dir:", MODELS_DIR.resolve())


Data path: /home/luis/Documents/mdms_pitt/courses/Capstone/projects/case-studies-capstone/data/processed/icu_stay_modeling_24h_v1.csv
Tables dir: /home/luis/Documents/mdms_pitt/courses/Capstone/projects/case-studies-capstone/outputs/tables
Figures dir: /home/luis/Documents/mdms_pitt/courses/Capstone/projects/case-studies-capstone/outputs/figures
Models dir: /home/luis/Documents/mdms_pitt/courses/Capstone/projects/case-studies-capstone/outputs/models


## 3) Load data and run basic integrity checks

In [3]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

assert "stay_id" in df.columns, "stay_id is missing"
assert "subject_id" in df.columns, "subject_id is missing"
assert "prolonged_los_8d" in df.columns, "target column prolonged_los_8d is missing"

assert df["stay_id"].notna().all(), "stay_id has nulls"
assert df["stay_id"].nunique() == len(df), "duplicate stay_id rows detected"
assert set(df["prolonged_los_8d"].dropna().unique()).issubset({0, 1}), "target must be 0/1"

print("\nTarget prevalence:")
print(df["prolonged_los_8d"].value_counts(dropna=False).sort_index())
print(df["prolonged_los_8d"].value_counts(normalize=True).sort_index().round(4))

stays_per_subject = df.groupby("subject_id")["stay_id"].nunique().sort_values(ascending=False)
repeat_subjects = stays_per_subject[stays_per_subject > 1]

print("\nSubjects with repeated ICU stays:", repeat_subjects.shape[0])
print("Max stays for a single subject:", int(stays_per_subject.max()))
repeat_subjects.head(10)


Shape: (140, 38)

Columns:
['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit', 'admission_type', 'intime', 'outtime', 'n_chartevents_6h', 'n_chartevents_24h', 'missing_core_vitals_24h_count', 'hr_first', 'hr_mean', 'hr_min', 'hr_max', 'hr_last', 'rr_first', 'rr_mean', 'rr_min', 'rr_max', 'rr_last', 'spo2_first', 'spo2_mean', 'spo2_min', 'spo2_max', 'spo2_last', 'temp_first', 'temp_mean', 'temp_min', 'temp_max', 'temp_last', 'map_first', 'map_mean', 'map_min', 'map_max', 'map_last', 'los_days', 'prolonged_los_8d']

Target prevalence:
prolonged_los_8d
0    124
1     16
Name: count, dtype: int64
prolonged_los_8d
0    0.8857
1    0.1143
Name: proportion, dtype: float64

Subjects with repeated ICU stays: 26
Max stays for a single subject: 5


subject_id
10020740    5
10002428    4
10014354    4
10017492    3
10019003    3
10015931    3
10023117    3
10016742    3
10039708    3
10003400    3
Name: stay_id, dtype: int64

## 4) Define target, predictors, and audit-only fields

The prepared dataset contains some columns that are useful for traceability or QA but should not be used for model fitting.  
For v1, `last_careunit` is treated as audit-only because it is too close to hindsight.


In [4]:
TARGET = "prolonged_los_8d"

AUDIT_ONLY_COLS = [
    "subject_id",
    "hadm_id",
    "stay_id",
    "intime",
    "outtime",
    "los_days",
    "last_careunit",
]

existing_audit_cols = [c for c in AUDIT_ONLY_COLS if c in df.columns]

X = df.drop(columns=[TARGET] + existing_audit_cols, errors="ignore").copy()
y = df[TARGET].astype(int).copy()
groups = df["subject_id"].astype(int).copy()

print("Predictor matrix shape:", X.shape)
print("Audit-only columns dropped:", existing_audit_cols)
print("\nPredictor columns:")
print(X.columns.tolist())


Predictor matrix shape: (140, 30)
Audit-only columns dropped: ['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'los_days', 'last_careunit']

Predictor columns:
['first_careunit', 'admission_type', 'n_chartevents_6h', 'n_chartevents_24h', 'missing_core_vitals_24h_count', 'hr_first', 'hr_mean', 'hr_min', 'hr_max', 'hr_last', 'rr_first', 'rr_mean', 'rr_min', 'rr_max', 'rr_last', 'spo2_first', 'spo2_mean', 'spo2_min', 'spo2_max', 'spo2_last', 'temp_first', 'temp_mean', 'temp_min', 'temp_max', 'temp_last', 'map_first', 'map_mean', 'map_min', 'map_max', 'map_last']


## 5) Helper functions

In [5]:
def make_onehot_encoder():
    params = {"handle_unknown": "ignore"}
    sig = inspect.signature(OneHotEncoder)
    if "sparse_output" in sig.parameters:
        params["sparse_output"] = False
    else:
        params["sparse"] = False
    return OneHotEncoder(**params)


def build_preprocessor(X_frame: pd.DataFrame) -> ColumnTransformer:
    cat_cols = [
        c for c in X_frame.columns
        if pd.api.types.is_object_dtype(X_frame[c]) or pd.api.types.is_categorical_dtype(X_frame[c])
    ]
    num_cols = [c for c in X_frame.columns if c not in cat_cols]

    numeric_pipe = Pipeline(steps=[
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])

    categorical_pipe = Pipeline(steps=[
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_onehot_encoder()),
    ])

    preprocess = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, num_cols),
            ("cat", categorical_pipe, cat_cols),
        ],
        remainder="drop",
    )
    return preprocess, num_cols, cat_cols


def make_group_holdout_split(X_frame, y_series, group_series, test_size=0.25, random_state=42):
    try:
        from sklearn.model_selection import StratifiedGroupKFold

        n_splits = int(round(1 / test_size))
        splitter = StratifiedGroupKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=random_state,
        )
        train_idx, test_idx = next(splitter.split(X_frame, y_series, group_series))
        strategy = f"StratifiedGroupKFold holdout (~{1/n_splits:.0%} test)"
    except Exception:
        splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
        train_idx, test_idx = next(splitter.split(X_frame, y_series, group_series))
        strategy = "GroupShuffleSplit holdout (fallback; not stratified)"
    return train_idx, test_idx, strategy


def make_group_cv(y_series, group_series, desired_splits=5, random_state=42):
    y_series = pd.Series(y_series)
    group_series = pd.Series(group_series)

    unique_groups = group_series.nunique()
    pos_groups = group_series[y_series == 1].nunique()
    neg_groups = group_series[y_series == 0].nunique()

    n_splits = min(desired_splits, unique_groups, pos_groups, neg_groups)
    if n_splits < 2:
        raise ValueError(
            f"Not enough groups to build a valid grouped CV splitter. "
            f"unique_groups={unique_groups}, pos_groups={pos_groups}, neg_groups={neg_groups}"
        )

    try:
        from sklearn.model_selection import StratifiedGroupKFold
        cv = StratifiedGroupKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=random_state,
        )
        cv_name = f"StratifiedGroupKFold ({n_splits} folds)"
    except Exception:
        cv = GroupKFold(n_splits=n_splits)
        cv_name = f"GroupKFold ({n_splits} folds fallback)"
    return cv, cv_name


def safe_metric(y_true, y_pred, y_prob):
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

    try:
        out["roc_auc"] = roc_auc_score(y_true, y_prob)
    except Exception:
        out["roc_auc"] = np.nan

    try:
        out["pr_auc"] = average_precision_score(y_true, y_prob)
    except Exception:
        out["pr_auc"] = np.nan

    try:
        out["brier"] = brier_score_loss(y_true, y_prob)
    except Exception:
        out["brier"] = np.nan

    return out


def evaluate_estimator(name, estimator, X_train, y_train, X_test, y_test):
    fitted = clone(estimator)
    fitted.fit(X_train, y_train)

    if hasattr(fitted, "predict_proba"):
        y_prob = fitted.predict_proba(X_test)[:, 1]
    else:
        y_prob = fitted.predict(X_test).astype(float)

    y_pred = fitted.predict(X_test)
    metrics = safe_metric(y_test, y_pred, y_prob)
    metrics["model"] = name

    pred_df = pd.DataFrame({
        "model": name,
        "y_true": y_test,
        "y_pred": y_pred,
        "y_prob": y_prob,
    })

    return fitted, metrics, pred_df


def save_figure(fig, path: Path):
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


## 6) Create the held-out split and save a split manifest

In [6]:
train_idx, test_idx, holdout_strategy = make_group_holdout_split(
    X_frame=X,
    y_series=y,
    group_series=groups,
    test_size=0.25,
    random_state=RANDOM_STATE,
)

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)
groups_train = groups.iloc[train_idx].reset_index(drop=True)
groups_test = groups.iloc[test_idx].reset_index(drop=True)

print("Holdout strategy:", holdout_strategy)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain prevalence:", y_train.mean().round(4))
print("Test prevalence:", y_test.mean().round(4))
print("Unique train subjects:", groups_train.nunique())
print("Unique test subjects:", groups_test.nunique())
print("Subjects shared across train/test:", len(set(groups_train).intersection(set(groups_test))))

assert len(set(groups_train).intersection(set(groups_test))) == 0, "Subject leakage across train/test split"

split_manifest = df[["stay_id", "subject_id", TARGET]].copy()
split_manifest["split"] = "train"
split_manifest.loc[test_idx, "split"] = "test"
split_manifest.to_csv(TABLES_DIR / "model_v1_split_manifest.csv", index=False)

split_manifest.head()


Holdout strategy: StratifiedGroupKFold holdout (~25% test)
Train shape: (102, 30)
Test shape: (38, 30)

Train prevalence: 0.1373
Test prevalence: 0.0526
Unique train subjects: 74
Unique test subjects: 26
Subjects shared across train/test: 0


,stay_id,subject_id,prolonged_los_8d,split
0,30057454,10023117,0,test
1,30101877,10032725,0,train
2,30425410,10016742,0,test
3,30458995,10031757,0,train
4,30585761,10022281,0,train


## 7) Build preprocessing and cross-validation objects

In [7]:
preprocess, num_cols, cat_cols = build_preprocessor(X_train)
cv, cv_name = make_group_cv(y_train, groups_train, desired_splits=5, random_state=RANDOM_STATE)

print("Grouped CV strategy:", cv_name)
print("\nNumeric columns:", num_cols)
print("\nCategorical columns:", cat_cols)


Grouped CV strategy: StratifiedGroupKFold (5 folds)

Numeric columns: ['n_chartevents_6h', 'n_chartevents_24h', 'missing_core_vitals_24h_count', 'hr_first', 'hr_mean', 'hr_min', 'hr_max', 'hr_last', 'rr_first', 'rr_mean', 'rr_min', 'rr_max', 'rr_last', 'spo2_first', 'spo2_mean', 'spo2_min', 'spo2_max', 'spo2_last', 'temp_first', 'temp_mean', 'temp_min', 'temp_max', 'temp_last', 'map_first', 'map_mean', 'map_min', 'map_max', 'map_last']

Categorical columns: ['first_careunit', 'admission_type']


/tmp/ipykernel_54319/418437241.py:14: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(X_frame[c]) or pd.api.types.is_categorical_dtype(X_frame[c])


## 8) Define model pipelines and tuning grids

The tuning is intentionally smallish. The point is not to run a giant search. The point is to give each model a fair enough chance without turning this into a 5 year project.


In [8]:
logreg_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=5000, solver="liblinear")),
])

rf_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
])

logreg_param_grid = {
    "model__C": [0.1, 1.0, 10.0],
    "model__class_weight": [None, "balanced"],
}

rf_param_grid = {
    "model__n_estimators": [300, 500],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_leaf": [1, 2, 5],
    "model__class_weight": [None, "balanced"],
}


## 9) Run model selection inside the training set only

In [9]:
search_scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

logreg_search = GridSearchCV(
    estimator=logreg_pipe,
    param_grid=logreg_param_grid,
    scoring=search_scoring,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

rf_search = GridSearchCV(
    estimator=rf_pipe,
    param_grid=rf_param_grid,
    scoring=search_scoring,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

logreg_search.fit(X_train, y_train, groups=groups_train)
rf_search.fit(X_train, y_train, groups=groups_train)

print("Best logistic regression params:")
print(logreg_search.best_params_)
print("Best logistic regression CV F1:", round(logreg_search.best_score_, 4))

print("\nBest random forest params:")
print(rf_search.best_params_)
print("Best random forest CV F1:", round(rf_search.best_score_, 4))


Fitting 5 folds for each of 6 candidates, totalling 30 fits


/home/luis/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/luis/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/luis/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/luis/.local/lib/python3.10/site-packages/sklea

Fitting 5 folds for each of 36 candidates, totalling 180 fits


/home/luis/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/luis/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/luis/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/luis/.local/lib/python3.10/site-packages/sklea

Best logistic regression params:
{'model__C': 10.0, 'model__class_weight': None}
Best logistic regression CV F1: 0.5591

Best random forest params:
{'model__class_weight': 'balanced', 'model__max_depth': None, 'model__min_samples_leaf': 5, 'model__n_estimators': 500}
Best random forest CV F1: 0.1238


## 10) Save model-search results

In [10]:
logreg_search_results = pd.DataFrame(logreg_search.cv_results_).sort_values("rank_test_f1")
rf_search_results = pd.DataFrame(rf_search.cv_results_).sort_values("rank_test_f1")

logreg_search_results.to_csv(TABLES_DIR / "model_v1_search_results_logreg.csv", index=False)
rf_search_results.to_csv(TABLES_DIR / "model_v1_search_results_rf.csv", index=False)

logreg_search_results[[
    "rank_test_f1",
    "mean_test_accuracy",
    "mean_test_precision",
    "mean_test_recall",
    "mean_test_f1",
    "mean_test_roc_auc",
    "param_model__C",
    "param_model__class_weight",
]].head()

rf_search_results[[
    "rank_test_f1",
    "mean_test_accuracy",
    "mean_test_precision",
    "mean_test_recall",
    "mean_test_f1",
    "mean_test_roc_auc",
    "param_model__n_estimators",
    "param_model__max_depth",
    "param_model__min_samples_leaf",
    "param_model__class_weight",
]].head()


,rank_test_f1,mean_test_accuracy,mean_test_precision,mean_test_recall,mean_test_f1,mean_test_roc_auc,param_model__n_estimators,param_model__max_depth,param_model__min_samples_leaf,param_model__class_weight
29,1,0.836471,0.133333,0.116667,0.123810,0.764160,500,5,5,balanced
23,1,0.836471,0.133333,0.116667,0.123810,0.764160,500,None,5,balanced
35,1,0.836471,0.133333,0.116667,0.123810,0.764160,500,10,5,balanced
22,4,0.826471,0.116667,0.116667,0.114286,0.773993,300,None,5,balanced
34,4,0.826471,0.116667,0.116667,0.114286,0.773993,300,10,5,balanced


## 11) Get cross-validation metrics for the chosen final configurations

In [11]:
cv_scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "brier": "neg_brier_score",
}

best_logreg = logreg_search.best_estimator_
best_rf = rf_search.best_estimator_

cv_results = []

for model_name, estimator in [
    ("Logistic regression", best_logreg),
    ("Random forest", best_rf),
]:
    scores = cross_validate(
        estimator,
        X_train,
        y_train,
        cv=cv,
        groups=groups_train,
        scoring=cv_scoring,
        return_train_score=False,
        error_score="raise",
        n_jobs=-1,
    )

    n_folds = len(scores["test_f1"])
    for fold in range(n_folds):
        cv_results.append({
            "model": model_name,
            "fold": fold + 1,
            "accuracy": scores["test_accuracy"][fold],
            "precision": scores["test_precision"][fold],
            "recall": scores["test_recall"][fold],
            "f1": scores["test_f1"][fold],
            "roc_auc": scores["test_roc_auc"][fold],
            "pr_auc": scores["test_pr_auc"][fold],
            "brier": -scores["test_brier"][fold],
        })

cv_metrics_by_fold = pd.DataFrame(cv_results)
cv_metrics_summary = (
    cv_metrics_by_fold
    .groupby("model")[["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "brier"]]
    .agg(["mean", "std", "min", "max"])
)

cv_metrics_by_fold.to_csv(TABLES_DIR / "model_v1_cv_metrics_by_fold.csv", index=False)
cv_metrics_summary.to_csv(TABLES_DIR / "model_v1_cv_metrics_summary.csv")

cv_metrics_summary


/home/luis/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/luis/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


accuracy                       precision                 \
                         mean       std   min   max      mean       std  min   
model                                                                          
Logistic regression  0.848235  0.120724  0.70  1.00  0.625714  0.359194  0.2   
Random forest        0.836471  0.079265  0.75  0.95  0.133333  0.182574  0.0   

                                 recall            ...   roc_auc       \
                          max      mean       std  ...       min  max   
model                                              ...                  
Logistic regression  1.000000  0.656667  0.373348  ...  0.533333  1.0   
Random forest        0.333333  0.116667  0.162447  ...  0.533333  1.0   

                       pr_auc                              brier            \
                         mean       std       min  max      mean       std   
model                                                                        
Logistic regression  0.521643  0.288362  0.234127  1.0  0.139408  0.098891   
Random forest        0.490334  0.287361  0.331650  1.0  0.127724  0.059143   

                                         
                          min       max  
model                                    
Logistic regression  0.013711  0.254205  
Random forest        0.064095  0.202599  

[2 rows x 28 columns]

## 12) Fit the final models on the training set and evaluate on held-out test data

In [12]:
baseline_clf = DummyClassifier(strategy="most_frequent")
baseline_clf.fit(X_train, y_train)

baseline_pred = baseline_clf.predict(X_test)
baseline_prob = baseline_clf.predict_proba(X_test)[:, 1]
baseline_metrics = safe_metric(y_test, baseline_pred, baseline_prob)
baseline_metrics["model"] = "Majority baseline"

fitted_logreg, logreg_metrics, logreg_pred_df = evaluate_estimator(
    name="Logistic regression",
    estimator=best_logreg,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
)

fitted_rf, rf_metrics, rf_pred_df = evaluate_estimator(
    name="Random forest",
    estimator=best_rf,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
)

test_metrics = pd.DataFrame([baseline_metrics, logreg_metrics, rf_metrics])
test_metrics = test_metrics[
    ["model", "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "brier"]
].sort_values("f1", ascending=False)

test_metrics.to_csv(TABLES_DIR / "model_v1_test_metrics.csv", index=False)
test_metrics.to_csv(TABLES_DIR / "model_v1_model_comparison.csv", index=False)

test_metrics


,model,accuracy,precision,recall,f1,roc_auc,pr_auc,brier
0,Majority baseline,0.947368,0.0,0.0,0.0,0.500000,0.052632,0.052632
1,Logistic regression,0.842105,0.0,0.0,0.0,0.750000,0.142857,0.113953
2,Random forest,0.894737,0.0,0.0,0.0,0.472222,0.071770,0.099737


## 13) Save held-out predictions and model files

In [13]:
predictions = df.iloc[test_idx][["stay_id", "subject_id", TARGET]].copy().reset_index(drop=True)
predictions = predictions.rename(columns={TARGET: "y_true"})

predictions["pred_majority_baseline"] = baseline_pred
predictions["prob_majority_baseline"] = baseline_prob

predictions["pred_logreg"] = logreg_pred_df["y_pred"].values
predictions["prob_logreg"] = logreg_pred_df["y_prob"].values

predictions["pred_random_forest"] = rf_pred_df["y_pred"].values
predictions["prob_random_forest"] = rf_pred_df["y_prob"].values

predictions.to_csv(TABLES_DIR / "model_v1_test_predictions.csv", index=False)

joblib.dump(fitted_logreg, MODELS_DIR / "model_v1_logreg.joblib")
joblib.dump(fitted_rf, MODELS_DIR / "model_v1_random_forest.joblib")

predictions.head()


,stay_id,subject_id,y_true,pred_majority_baseline,prob_majority_baseline,pred_logreg,prob_logreg,pred_random_forest,prob_random_forest
0,30057454,10023117,0,0,0.0,0,0.000019,0,0.167451
1,30425410,10016742,0,0,0.0,0,0.022782,0,0.343442
2,30665396,10018423,0,0,0.0,0,0.013918,0,0.306781
3,30955999,10023117,1,0,0.0,0,0.072759,0,0.156257
4,31077365,10020740,0,0,0.0,0,0.000006,0,0.382249


## 14) ROC and precision-recall curves

In [14]:
# ROC comparison
fig, ax = plt.subplots()
for label, probs in [
    ("Logistic regression", logreg_pred_df["y_prob"].values),
    ("Random forest", rf_pred_df["y_prob"].values),
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = roc_auc_score(y_test, probs)
    ax.plot(fpr, tpr, label=f"{label} (AUC={roc_auc:.3f})")

ax.plot([0, 1], [0, 1], linestyle="--")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Held-out ROC Curve Comparison")
ax.legend(loc="lower right")

save_figure(fig, FIGURES_DIR / "model_v1_roc_curve_comparison.png")

# Precision-recall comparison
fig, ax = plt.subplots()
prevalence = y_test.mean()
for label, probs in [
    ("Logistic regression", logreg_pred_df["y_prob"].values),
    ("Random forest", rf_pred_df["y_prob"].values),
]:
    precision, recall, _ = precision_recall_curve(y_test, probs)
    ap = average_precision_score(y_test, probs)
    ax.plot(recall, precision, label=f"{label} (AP={ap:.3f})")

ax.axhline(prevalence, linestyle="--", label=f"Prevalence={prevalence:.3f}")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Held-out Precision-Recall Curve Comparison")
ax.legend(loc="best")

save_figure(fig, FIGURES_DIR / "model_v1_precision_recall_curve_comparison.png")


## 15) Confusion matrices

In [15]:
for label, preds, filename in [
    ("Logistic regression", logreg_pred_df["y_pred"].values, "model_v1_confusion_matrix_logreg.png"),
    ("Random forest", rf_pred_df["y_pred"].values, "model_v1_confusion_matrix_rf.png"),
]:
    fig, ax = plt.subplots()
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(f"Held-out Confusion Matrix — {label}")
    save_figure(fig, FIGURES_DIR / filename)


## 16) Logistic regression coefficients

In [16]:
feature_names_logreg = fitted_logreg.named_steps["preprocess"].get_feature_names_out()
logreg_coefs = fitted_logreg.named_steps["model"].coef_.ravel()

logreg_coef_df = pd.DataFrame({
    "feature": feature_names_logreg,
    "coef": logreg_coefs,
})
logreg_coef_df["abs_coef"] = logreg_coef_df["coef"].abs()
logreg_coef_df = logreg_coef_df.sort_values("abs_coef", ascending=False)

logreg_coef_df.to_csv(TABLES_DIR / "model_v1_logreg_coefficients.csv", index=False)

plot_df = pd.concat([
    logreg_coef_df.sort_values("coef", ascending=False).head(10),
    logreg_coef_df.sort_values("coef", ascending=True).head(10),
]).drop_duplicates()

fig, ax = plt.subplots(figsize=(9, 7))
plot_df = plot_df.sort_values("coef")
ax.barh(plot_df["feature"], plot_df["coef"])
ax.set_title("Logistic Regression Coefficients (Top Positive / Negative)")
ax.set_xlabel("Coefficient")

save_figure(fig, FIGURES_DIR / "model_v1_logreg_coefficients.png")

logreg_coef_df.head(15)


,feature,coef,abs_coef
28,cat__first_careunit_Cardiac Vascular Intensive...,-4.439930,4.439930
4,num__hr_mean,2.774075,2.774075
3,num__hr_first,-2.713590,2.713590
1,num__n_chartevents_24h,2.606130,2.606130
5,num__hr_min,-2.286305,2.286305
35,cat__first_careunit_Surgical Intensive Care Un...,-2.254423,2.254423
13,num__spo2_first,2.122464,2.122464
27,num__map_last,-1.682989,1.682989
24,num__map_mean,1.600482,1.600482
36,cat__first_careunit_Trauma SICU (TSICU),1.339112,1.339112


## 17) Random forest feature importance

In [17]:
feature_names_rf = fitted_rf.named_steps["preprocess"].get_feature_names_out()
rf_importances = fitted_rf.named_steps["model"].feature_importances_

rf_importance_df = pd.DataFrame({
    "feature": feature_names_rf,
    "importance": rf_importances,
}).sort_values("importance", ascending=False)

rf_importance_df.to_csv(TABLES_DIR / "model_v1_rf_feature_importance.csv", index=False)

plot_df = rf_importance_df.head(20).sort_values("importance")
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(plot_df["feature"], plot_df["importance"])
ax.set_title("Random Forest Feature Importance (Top 20)")
ax.set_xlabel("Importance")

save_figure(fig, FIGURES_DIR / "model_v1_rf_feature_importance.png")

rf_importance_df.head(20)


,feature,importance
1,num__n_chartevents_24h,0.130344
21,num__temp_max,0.073425
19,num__temp_mean,0.067540
14,num__spo2_mean,0.062222
9,num__rr_mean,0.061936
10,num__rr_min,0.049388
27,num__map_last,0.040878
25,num__map_min,0.039534
22,num__temp_last,0.039039
6,num__hr_max,0.037667
